# 1. Import Necessary Packages

In [ ]:
# Uncomment and run the following line if required packages are not installed
# !pip install pandas numpy scipy statsmodels matplotlib seaborn openpyxl

# Core data handling
import pandas as pd        # Tabular data structures (DataFrame), I/O, groupby/merge, etc.
import numpy as np         # Numerical computing: arrays, vectorization, stats helpers

# Plotting
import matplotlib.pyplot as plt  # Base plotting (figures, axes, annotations, savefig)
import seaborn as sns            # Statistical/beautiful plots built on matplotlib (themes, high-level charts)

# File & paths
import os                        # File system ops (paths, mkdir, exists), environment variables

# Statistics / tests
from scipy.stats import mannwhitneyu
from scipy import stats 

# Others
from matplotlib.backends.backend_pdf import PdfPages  # Save multiple figures into a single PDF
import matplotlib.image as mpimg                      # Read/display images (e.g., when assembling figure grids)

import matplotlib as mpl



# 2. Set up

## 2.1 Set Up Working Path

In [ ]:
os.getcwd()
# Set global paths
PROJECT_PATH = r"C:\Your\Replication\Folder"

# subfolders
DATA_PATH   = os.path.join(PROJECT_PATH, "Data")
RESULT_PATH = os.path.join(PROJECT_PATH, "Result")

# Print paths
print("Project Path:", PROJECT_PATH)
print("Data Path:", DATA_PATH)
print("Result Path:", RESULT_PATH)

# Optionally change the working directory to one of them
os.chdir(DATA_PATH)
#os.chdir(RESULT_PATH)

## 2.2 Set Up Formatting

In [ ]:
# Apply Nature-style formatting globally
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.8,
    'figure.dpi': 600,
})

# Confirm update
print("Matplotlib parameters updated to Nature Food journal style.")

# Set font sizes for readability
plt.rcParams.update({
    'axes.titlesize': 18,
    'axes.labelsize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 14,
    'legend.fontsize': 10,
    'figure.titlesize': 8
})


## 2.3 Load Dataset

In [ ]:
os.chdir(DATA_PATH)
# Load the dataset
df = pd.read_excel("GNPD-AllFoodDrink_Claim_NPMScore_2015_2024_synthetic.xlsx", engine="openpyxl")

# 3. Results: Appendix

## Supplementary Fig.S1 | Claims Popularity by Product Category and Claim Types. 

In [ ]:

# Set font sizes for better readability
plt.rcParams.update({
    'axes.titlesize': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 18
})

# Define claim variables and labels
claim_vars = ['Num_Minus', 'Num_Plus', 'Num_Natural', 'Num_Functional']
claim_labels = ['Minus', 'Plus', 'Natural', 'Functional']

# Define color map from style_map_color
style_map_color = {
    'Minus': '#DD8452',
    'Plus': '#8172B3',
    'Natural': '#4C72B0',
    'Functional': '#da8bc3'
}

# Ensure Category is string type
df['Category'] = df['NewCategory'].astype(str)

# Calculate total number of products per category
summary = df.groupby('Category').agg({'RecordID': 'count'}).rename(columns={'RecordID': 'TotalProducts'})
product_counts = summary['TotalProducts']
category_order = product_counts.index.tolist()

# Initialize dictionaries to store results for share plot
share_data = {}
ci_share_data = {}

# Compute share and confidence intervals
for claim, label in zip(claim_vars, claim_labels):
    df[f'{label}_has_claim'] = df[claim] > 0
    grouped = df.groupby('Category')[f'{label}_has_claim']
    share = grouped.mean()
    n = grouped.count()
    ci = 1.96 * np.sqrt((share * (1 - share)) / n)

    share_data[label] = share * 100  # convert to percentage
    ci_share_data[label] = ci * 100  # convert to percentage

# Create combined figure
fig, axes = plt.subplots(2, 4, figsize=(24, 15))

# Plot share of products (top row)
for i, label in enumerate(claim_labels):
    ax = axes[0, i]
    x_labels = category_order
    x = np.arange(len(x_labels))
    y = share_data[label].reindex(x_labels)
    ci = ci_share_data[label].reindex(x_labels)
    sizes = product_counts.reindex(x_labels) / 50

    ax.errorbar(x, y, yerr=ci, fmt='o', markersize=0, ecolor='gray', capsize=3)
    ax.scatter(x, y, s=sizes, alpha=0.7, label=label, color=style_map_color[label])
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=90)
    ax.set_title(f"{label} Claim")
    ax.set_ylim(0, 100)
    ax.grid(False)
    if i == 0:
        ax.set_ylabel("Share of Products with Claims (%)")

    # Add share labels above bubbles with dynamic spacing
    for xi, yi, si in zip(x, y, sizes):
        ax.text(xi, yi + si**0.5 * 0.6, f"{yi:.1f}", ha='center', va='bottom', fontsize=10)

# Determine global y-axis limits for violin plots
y_min = float('inf')
y_max = float('-inf')
for var in claim_vars:
    filtered_df = df[df[var] > 0]
    if not filtered_df.empty:
        y_min = min(y_min, filtered_df[var].min())
        y_max = max(y_max, filtered_df[var].max())

# Plot violin plots (bottom row)
for i, label in enumerate(claim_labels):
    ax = axes[1, i]
    filtered_df = df[df[claim_vars[i]] > 0].copy()
    filtered_df['Category'] = pd.Categorical(filtered_df['Category'], categories=category_order, ordered=True)

    sns.violinplot(
        data=filtered_df,
        x='Category',
        y=claim_vars[i],
        ax=ax,
        palette=[style_map_color[label]],
        cut=0,
        inner=None,
        linewidth=1
    )

    # Overlay mean and 95% CI
    grouped = filtered_df.groupby('Category')[claim_vars[i]]
    means = grouped.mean()
    stds = grouped.std()
    counts = grouped.count()
    ci95 = 1.96 * stds / np.sqrt(counts)

    x_pos = np.arange(len(category_order))
    ax.scatter(x_pos, means.reindex(category_order), color='black', zorder=3, label='Mean')
    ax.errorbar(x_pos, means.reindex(category_order), yerr=ci95.reindex(category_order), fmt='none', ecolor='black', capsize=3)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(category_order, rotation=90)
    ax.set_title(f"{label} Claim")
    ax.set_ylim(y_min, y_max)
    ax.grid(False)
    if i == 0:
        ax.set_ylabel("Number of Claims per Product")
    else:
        ax.set_ylabel("")

# Set overall title and layout
#fig.suptitle("Claims Popularity by Category\n(Dot Size = Number of New Products, Error Bars = 95% CI\nTop: Share of Products with Claims | Bottom: Number of Claims per Product (Violin Plot))", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# Export results
os.chdir(RESULT_PATH)
# Define output file name
output_filename = "AppendixFigure1_ClaimsPopularityFoodDrink_ClaimType.png"
output_path = os.path.join(RESULT_PATH, output_filename)
# Remove the file if it already exists
if os.path.exists(output_path):
    os.remove(output_path)
# Save and show the figure
plt.savefig(output_path, dpi=1200, bbox_inches='tight')
plt.show()


## Supplementary Fig.S2 | Discrepancy of Claims by Category: Food

In [ ]:
import os
import re
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker
from scipy.stats import mannwhitneyu
from tqdm import tqdm

os.makedirs(RESULT_PATH, exist_ok=True)

# ============================================================
# Output paths
# ============================================================
food_appendix_folder = os.path.join(RESULT_PATH, "Food_Category_Panels")
os.makedirs(food_appendix_folder, exist_ok=True)

single_panel_folder = os.path.join(food_appendix_folder, "Single_Category_Figures")
os.makedirs(single_panel_folder, exist_ok=True)

# ============================================================
# Claim groups and labels
# ============================================================
claim_groups = {
    'Minus': [
        'LessCalorie', 'NoAddedSugar', 'SugarFree', 'LowSugar', 'Diet',
        'LessSodium', 'LessCarb', 'LessFat', 'LessTransFat', 'LessSatFat',
        'LessChol', 'LessGlycemic'
    ],
    'Plus': [
        'PlusVitamin', 'HighProtein', 'AddedCalcium', 'HighFiber'
    ],
    'Natural': [
        'AllNatural', 'NoArtAdditives', 'NoArtColourings', 'NoArtFlavourings',
        'NoArtPreservatives', 'NoAdditivesPreservatives', 'GMOFree', 'Organic', 'Wholegrain'
    ],
    'Functional': [
        'BrainNervSystem', 'ImmuneSystem', 'Digestive', 'Probiotic',
        'Antioxidant', 'WgtMuscle', 'Cardiovascular', 'BoneSkinHairEyeHealth'
    ]
}

claim_labels = [
    'Low/No/Reduced Calorie', 'No Added Sugar', 'Sugar Free', 'Low/Reduced Sugar', 'Diet/Light',
    'Low/No/Reduced Sodium', 'Low/No/Reduced Carb', 'Low/No/Reduced Fat', 'Low/No/Reduced Trans Fat',
    'Low/No/Reduced Saturated Fat', 'Low/No/Reduced Cholesterol', 'Low/No/Reduced Glycemic',
    'Vitamin/Mineral Fortified', 'High/Added Protein', 'Added Calcium',
    'High/Added Fiber', 'All Natural Product', 'No Added/Artificial Additives',
    'No Added/Artificial Colourings', 'No Added/Artificial Flavourings', 'No Added/Artificial Preservatives',
    'No Additives/Preservatives', 'GMO Free', 'Organic', 'Whole Grain',
    'Brain & Nervous System', 'Immune System', 'Digestive', 'Probiotic/Prebiotic',
    'Antioxidant', 'Weight & Muscle Gain', 'Cardiovascular', 'Bone/Skin/Nails&Hair/Eye Health'
]

claims = [c for grp in claim_groups.values() for c in grp]
claim_label_map = dict(zip(claims, claim_labels))

# ============================================================
# Helpers
# ============================================================
def safe_name(s):
    return re.sub(r'[^a-zA-Z0-9._-]+', '_', str(s).strip())

def get_significance_marker(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''

def assign_group(claim):
    for group, items in claim_groups.items():
        if claim in items:
            return group
    return 'Other'

def healthy_text_for_category(category):
    return "(<4 = Healthier under NPM; ≥4 = Less healthy under NPM)", 4.0

# ============================================================
# Core computation
# ============================================================
def compute_results_for_subset(df_subset):
    """
    Build results for one food category.
    Uses Wilcoxon rank-sum / Mann–Whitney U.
    Distinguishes 'not testable' from 'not significant'.
    """
    if df_subset.empty:
        return pd.DataFrame(), pd.DataFrame()

    rows = []

    for claim in claims:
        if claim not in df_subset.columns:
            continue

        with_claim = df_subset.loc[df_subset[claim] == 1, 'np_score'].dropna()
        without_claim = df_subset.loc[df_subset[claim] == 0, 'np_score'].dropna()

        mean_with = with_claim.mean() if len(with_claim) else np.nan
        mean_without = without_claim.mean() if len(without_claim) else np.nan
        popularity = df_subset[claim].mean() * 100 if claim in df_subset.columns else np.nan

        if len(with_claim) == 0 or len(without_claim) == 0:
            p_val = np.nan
            status = 'Not testable'
        else:
            try:
                _, p_val = mannwhitneyu(
                    with_claim,
                    without_claim,
                    alternative='two-sided'
                )
            except ValueError:
                p_val = np.nan

            if pd.isna(p_val):
                status = 'Not testable'
            else:
                status = 'Testable'

        rows.append({
            'Claim': claim,
            'With Claim': mean_with,
            'Without Claim': mean_without,
            'Popularity': popularity,
            'p_value': p_val,
            'Status': status
        })

    results_df = pd.DataFrame(rows)
    if results_df.empty:
        return results_df, results_df

    results_df['Significance'] = results_df['p_value'].apply(get_significance_marker)
    results_df['Direction'] = results_df['With Claim'] - results_df['Without Claim']
    results_df['Group'] = results_df['Claim'].apply(assign_group)

    deep_palette = sns.color_palette('deep')
    blue_deep = deep_palette[0]
    orange_deep = deep_palette[1]
    gray_light = '#BDBDBD'

    def choose_color(row):
        if row['Status'] == 'Not testable':
            return gray_light
        if pd.notna(row['Direction']) and pd.notna(row['p_value']) and row['p_value'] < 0.05:
            if row['Direction'] > 0:
                return orange_deep
            elif row['Direction'] < 0:
                return blue_deep
        return 'black'

    results_df['Color'] = results_df.apply(choose_color, axis=1)

    def make_label(row):
        base = claim_label_map[row['Claim']]
        if row['Status'] == 'Not testable':
            return base
        if pd.notna(row['p_value']) and row['p_value'] < 0.05:
            return f"{base} {row['Significance']}"
        return base

    results_df['Label'] = results_df.apply(make_label, axis=1)
    results_df['Flag'] = results_df.apply(
        lambda r: bool(
            r['Status'] == 'Testable' and
            pd.notna(r['Direction']) and
            pd.notna(r['p_value']) and
            r['Direction'] > 0 and
            r['p_value'] < 0.05
        ),
        axis=1
    )

    results_melted = results_df.melt(
        id_vars=['Claim', 'Popularity', 'p_value', 'Label', 'Color', 'Flag', 'Group', 'Status'],
        value_vars=['With Claim', 'Without Claim'],
        var_name='Claim Status',
        value_name='Average NPM Score'
    )

    label_order = results_df['Label'].tolist()
    results_melted['Label'] = pd.Categorical(
        results_melted['Label'],
        categories=label_order,
        ordered=True
    )

    return results_df, results_melted

# ============================================================
# Drawing
# ============================================================
def draw_one_panel(ax, results_df, results_melted,
                   title=None, show_legend=False,
                   healthy_rule_text="(<4 = Healthier under NPM; ≥4 = Less healthy under NPM)"):
    if results_df.empty or results_melted.empty:
        ax.set_axis_off()
        if title:
            ax.set_title(f"{title}\n(no data)", fontsize=12, pad=4)
        return

    sns.set_theme(style="white")
    ax.set_axisbelow(True)

    sns.scatterplot(
        data=results_melted,
        x='Average NPM Score',
        y='Label',
        hue='Claim Status',
        size='Popularity',
        sizes=(50, 400),
        palette='deep',
        style='Claim Status',
        legend='brief' if show_legend else False,
        ax=ax,
        zorder=2
    )

    label_color_map = dict(zip(results_df['Label'], results_df['Color']))
    for tick in ax.get_yticklabels():
        txt = tick.get_text()
        if txt in label_color_map:
            tick.set_color(label_color_map[txt])

    label_order = results_df['Label'].tolist()
    label_to_group = dict(zip(results_df['Label'], results_df['Group']))
    spans, prev_g, start_i = [], None, None

    for i, lab in enumerate(label_order):
        g = label_to_group.get(lab, 'Other')
        if g != prev_g:
            if prev_g is not None:
                spans.append((start_i, i, prev_g))
            start_i = i
            prev_g = g
    spans.append((start_i, len(label_order), prev_g))

    x0, x1 = ax.get_xlim()
    for j, (s, e, g) in enumerate(spans):
        y0 = s - 0.5
        height = e - s
        if j != 0 and j != len(spans) - 1:
            ax.add_patch(
                patches.Rectangle(
                    (x0, y0),
                    width=(x1 - x0),
                    height=height,
                    fill=False,
                    edgecolor='black',
                    linewidth=1.0,
                    zorder=1,
                    clip_on=True
                )
            )

    n_positions = max(len(label_order) - 1, 1)
    for (s, e, g) in spans:
        y_center_frac = ((s + e - 1) / 2) / n_positions
        ax.text(
            -0.62, y_center_frac, g,
            transform=ax.transAxes,
            ha='right', va='center',
            fontsize=9, rotation=90,
            fontweight='bold', color='black',
            clip_on=False
        )

    ax.tick_params(axis='x', labelsize=9)
    ax.tick_params(axis='y', labelsize=9)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))

    ax.set_xlabel(f"Average Nutrition Profile Score\n{healthy_rule_text}", fontsize=9)
    ax.set_ylabel("Claims", fontweight='bold', labelpad=15, fontsize=9)

    if title:
        ax.set_title(title, fontsize=12, pad=4)

    ax.grid(True, which='both', axis='both',
            linestyle='--', linewidth=0.8, alpha=0.65, zorder=0)

# ============================================================
# Save single figure per food category
# ============================================================
def save_single_category_figure(category, df_cat):
    results_df, results_melted = compute_results_for_subset(df_cat)

    healthy_text, _ = healthy_text_for_category(category)

    n_labels = len(results_df) if not results_df.empty else 0
    fig_height = max(6, min(14, 0.35 * max(n_labels, 1) + 2))
    fig, ax = plt.subplots(figsize=(9, fig_height))

    draw_one_panel(
        ax, results_df, results_melted,
        title=f"{category}: Discrepancy in NP Scores by Claim",
        show_legend=True,
        healthy_rule_text=healthy_text
    )

    handles, labels = ax.get_legend_handles_labels()
    filtered = [(h, l) for h, l in zip(handles, labels) if l in ['With Claim', 'Without Claim']]
    if filtered:
        h, l = zip(*filtered)
        ax.legend(
            h, l,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.08),
            ncol=2,
            fontsize=10,
            frameon=True
        )
    else:
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

    fig.tight_layout()
    fig.subplots_adjust(left=0.2)

    out_path = os.path.join(
        single_panel_folder,
        f"Food_Category_Discrepancy_{safe_name(category)}.png"
    )
    fig.savefig(out_path, dpi=600, bbox_inches='tight')
    plt.close(fig)

# ============================================================
# Save combined portrait pages: 2 columns × 3 rows
# ============================================================
def save_combined_panels(page_categories, page_idx, total_pages, df_food):
    rows, cols = 3, 2
    fig, axes = plt.subplots(rows, cols, figsize=(12, 18), sharex=False, sharey=False)
    axes = axes.flatten()

    for i in range(rows * cols):
        ax = axes[i]

        if i >= len(page_categories):
            ax.set_axis_off()
            continue

        cat = page_categories[i]
        df_cat = df_food.loc[df_food['NewCategory'] == cat].copy()
        results_df, results_melted = compute_results_for_subset(df_cat)
        healthy_text, _ = healthy_text_for_category(cat)

        draw_one_panel(
            ax, results_df, results_melted,
            title=cat,
            show_legend=False,
            healthy_rule_text=healthy_text
        )

    # layout first, then put legend just below the last row dynamically
    bottom_rect = 0.10 if page_idx == 0 else 0.14
    fig.tight_layout(rect=[0, bottom_rect, 1, 0.985])

    fig.subplots_adjust(
        left=0.13,
        right=0.98,
        top=0.97,
        bottom=bottom_rect + 0.01,
        hspace=0.28,
        wspace=0.80
    )

    # determine bottom of subplot grid dynamically
    used_axes = [ax for i, ax in enumerate(axes) if i < len(page_categories)]
    grid_bottom = min(ax.get_position().y0 for ax in used_axes) if used_axes else 0.12

    # dummy legend
    dummy = pd.DataFrame({
        'Average NPM Score': [0, 0],
        'Label': ['a', 'b'],
        'Claim Status': ['With Claim', 'Without Claim'],
        'Popularity': [1, 1]
    })
    dummy_ax = fig.add_subplot(111, frame_on=False)
    dummy_ax.set_axis_off()
    lg = sns.scatterplot(
        data=dummy, x='Average NPM Score', y='Label',
        hue='Claim Status', style='Claim Status', ax=dummy_ax
    )
    handles, labels = lg.get_legend_handles_labels()

    fig.legend(
        handles, labels,
        loc='upper center',
        ncol=2,
        fontsize=11,
        frameon=True,
        borderaxespad=0,
        bbox_to_anchor=(0.5, grid_bottom - 0.03)
    )
    dummy_ax.remove()

    out_path = os.path.join(
        food_appendix_folder,
        f"Food_Category_Discrepancy_Combined_Page_{page_idx + 1}_of_{total_pages}.png"
    )
    fig.savefig(out_path, dpi=600, bbox_inches='tight')
    plt.close(fig)

# ============================================================
# MAIN
# ============================================================
# exclude beverages
df_food = df.loc[df['NewCategory'] != 'Beverages'].copy()

categories = (
    df_food['NewCategory']
    .dropna()
    .astype(str)
    .sort_values()
    .unique()
    .tolist()
)

print("Food categories included:")
for c in categories:
    print(" -", c)

# Save single-category figures
for cat in tqdm(categories, desc="Generating single food category figures"):
    df_cat = df_food.loc[df_food['NewCategory'] == cat].copy()
    save_single_category_figure(cat, df_cat)

# Save combined portrait pages: 2 per row, 3 rows per page
rows_per_page = 3
cols_per_page = 2
page_size = rows_per_page * cols_per_page

category_pages = [
    categories[i:i + page_size]
    for i in range(0, len(categories), page_size)
]

total_pages = len(category_pages)

for page_idx, page_categories in tqdm(
    list(enumerate(category_pages)),
    desc="Building combined food category pages",
    total=total_pages
):
    save_combined_panels(
        page_categories=page_categories,
        page_idx=page_idx,
        total_pages=total_pages,
        df_food=df_food
    )

print("\nDone.")
print(f"Single-category figures saved to: {single_panel_folder}")
print(f"Combined portrait pages saved to: {food_appendix_folder}")

## Supplementary Fig.S3 | Discrepancy of Claims by Category: Beverages. 

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker
from scipy.stats import mannwhitneyu
from tqdm import tqdm

os.makedirs(RESULT_PATH, exist_ok=True)

# ============================================================
# Beverage NewSubCategory appendix figures (S3)
# ============================================================

beverage_appendix_folder = os.path.join(RESULT_PATH, "Beverage_NewSubCategory_Panels")
os.makedirs(beverage_appendix_folder, exist_ok=True)

single_panel_folder = os.path.join(beverage_appendix_folder, "Single_NewSubCategory_Figures")
os.makedirs(single_panel_folder, exist_ok=True)

# ============================================================
# Claim groups & labels
# ============================================================

claim_groups = {
    'Minus': [
        'LessCalorie', 'NoAddedSugar', 'SugarFree', 'LowSugar', 'Diet',
        'LessSodium', 'LessCarb', 'LessFat', 'LessTransFat', 'LessSatFat',
        'LessChol', 'LessGlycemic'
    ],
    'Plus': [
        'PlusVitamin', 'HighProtein', 'AddedCalcium', 'HighFiber'
    ],
    'Natural': [
        'AllNatural', 'NoArtAdditives', 'NoArtColourings', 'NoArtFlavourings',
        'NoArtPreservatives', 'NoAdditivesPreservatives', 'GMOFree', 'Organic', 'Wholegrain'
    ],
    'Functional': [
        'BrainNervSystem', 'ImmuneSystem', 'Digestive', 'Probiotic',
        'Antioxidant', 'WgtMuscle', 'Cardiovascular', 'BoneSkinHairEyeHealth'
    ]
}

claim_labels = [
    'Low/No/Reduced Calorie', 'No Added Sugar', 'Sugar Free', 'Low/Reduced Sugar', 'Diet/Light',
    'Low/No/Reduced Sodium', 'Low/No/Reduced Carb', 'Low/No/Reduced Fat', 'Low/No/Reduced Trans Fat',
    'Low/No/Reduced Saturated Fat', 'Low/No/Reduced Cholesterol', 'Low/No/Reduced Glycemic',
    'Vitamin/Mineral Fortified', 'High/Added Protein', 'Added Calcium',
    'High/Added Fiber', 'All Natural Product', 'No Added/Artificial Additives',
    'No Added/Artificial Colourings', 'No Added/Artificial Flavourings', 'No Added/Artificial Preservatives',
    'No Additives/Preservatives', 'GMO Free', 'Organic', 'Whole Grain',
    'Brain & Nervous System', 'Immune System', 'Digestive', 'Probiotic/Prebiotic',
    'Antioxidant', 'Weight & Muscle Gain', 'Cardiovascular', 'Bone/Skin/Nails&Hair/Eye Health'
]

claims = [c for grp in claim_groups.values() for c in grp]
claim_label_map = dict(zip(claims, claim_labels))

# ============================================================
# Helpers
# ============================================================

def safe_name(s: str) -> str:
    return re.sub(r'[^a-zA-Z0-9._-]+', '_', str(s).strip())

def get_significance_marker(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''

def assign_group(claim):
    for group, items in claim_groups.items():
        if claim in items:
            return group
    return 'Other'

def healthy_text_for_beverage_subcategory():
    return "(<1 = Healthier under NPM; ≥1 = Less healthy under NPM)", 1.0

# ============================================================
# Core computation
# ============================================================

def compute_results_for_subset(df_subset: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:

    if df_subset.empty:
        return pd.DataFrame(), pd.DataFrame()

    rows = []

    for claim in claims:
        if claim not in df_subset.columns:
            continue

        with_claim = df_subset.loc[df_subset[claim] == 1, 'np_score'].dropna()
        without_claim = df_subset.loc[df_subset[claim] == 0, 'np_score'].dropna()

        popularity = df_subset[claim].mean() * 100

        if len(with_claim) == 0 or len(without_claim) == 0:
            p_val = np.nan
            status = 'Not applicable'
        else:
            try:
                _, p_val = mannwhitneyu(
                    with_claim,
                    without_claim,
                    alternative='two-sided'
                )
                status = 'Testable'
            except ValueError:
                p_val = np.nan
                status = 'Not applicable'

        rows.append({
            'Claim': claim,
            'With Claim': with_claim.mean() if len(with_claim) else np.nan,
            'Without Claim': without_claim.mean() if len(without_claim) else np.nan,
            'Popularity': popularity,
            'p_value': p_val,
            'Status': status
        })

    results_df = pd.DataFrame(rows)

    if results_df.empty:
        return results_df, results_df

    results_df['Significance'] = results_df['p_value'].apply(get_significance_marker)
    results_df['Direction'] = results_df['With Claim'] - results_df['Without Claim']
    results_df['Group'] = results_df['Claim'].apply(assign_group)

    deep_palette = sns.color_palette('deep')
    blue_deep = deep_palette[0]
    orange_deep = deep_palette[1]
    gray_light = '#BDBDBD'

    def choose_color(row):
        if row['Status'] == 'Not applicable':
            return gray_light
        if pd.notna(row['Direction']) and pd.notna(row['p_value']) and row['p_value'] < 0.05:
            if row['Direction'] > 0:
                return orange_deep
            elif row['Direction'] < 0:
                return blue_deep
        return 'black'

    results_df['Color'] = results_df.apply(choose_color, axis=1)

    def make_label(row):
        base = claim_label_map[row['Claim']]
        if row['Status'] == 'Not applicable':
            return base
        if pd.notna(row['p_value']) and row['p_value'] < 0.05:
            return f"{base} {row['Significance']}"
        return base

    results_df['Label'] = results_df.apply(make_label, axis=1)

    results_df['Flag'] = results_df.apply(
        lambda r: bool(
            r['Status'] == 'Testable' and
            pd.notna(r['Direction']) and
            pd.notna(r['p_value']) and
            r['Direction'] > 0 and
            r['p_value'] < 0.05
        ),
        axis=1
    )

    results_melted = results_df.melt(
        id_vars=[
            'Claim', 'Popularity', 'p_value', 'Label',
            'Color', 'Flag', 'Group', 'Status'
        ],
        value_vars=['With Claim', 'Without Claim'],
        var_name='Claim Status',
        value_name='Average NPM Score'
    )

    label_order = results_df['Label'].tolist()

    results_melted['Label'] = pd.Categorical(
        results_melted['Label'],
        categories=label_order,
        ordered=True
    )

    return results_df, results_melted

# ============================================================
# Drawing
# ============================================================

def draw_one_panel(
    ax,
    results_df: pd.DataFrame,
    results_melted: pd.DataFrame,
    title: str = None,
    show_legend=False,
    healthy_rule_text="(<1 = Healthier under NPM; ≥1 = Less healthy under NPM)"
):

    if results_df.empty or results_melted.empty:
        ax.set_axis_off()
        if title:
            ax.set_title(f"{title}\n(no data)", fontsize=12, pad=4)
        return

    sns.set_theme(style="white")
    ax.set_axisbelow(True)

    sns.scatterplot(
        data=results_melted,
        x='Average NPM Score',
        y='Label',
        hue='Claim Status',
        size='Popularity',
        sizes=(50, 400),
        palette='deep',
        style='Claim Status',
        legend='brief' if show_legend else False,
        ax=ax,
        zorder=2
    )

    label_color_map = dict(zip(results_df['Label'], results_df['Color']))

    for tick in ax.get_yticklabels():
        txt = tick.get_text()
        if txt in label_color_map:
            tick.set_color(label_color_map[txt])

    label_order = results_df['Label'].tolist()
    label_to_group = dict(zip(results_df['Label'], results_df['Group']))

    spans, prev_g, start_i = [], None, None

    for i, lab in enumerate(label_order):
        g = label_to_group.get(lab, 'Other')
        if g != prev_g:
            if prev_g is not None:
                spans.append((start_i, i, prev_g))
            start_i = i
            prev_g = g

    spans.append((start_i, len(label_order), prev_g))

    x0, x1 = ax.get_xlim()

    for j, (s, e, g) in enumerate(spans):
        y0 = s - 0.5
        height = e - s

        if j != 0 and j != len(spans) - 1:
            ax.add_patch(
                patches.Rectangle(
                    (x0, y0),
                    width=(x1 - x0),
                    height=height,
                    fill=False,
                    edgecolor='black',
                    linewidth=1.0,
                    zorder=1,
                    clip_on=True
                )
            )

    n_positions = max(len(label_order) - 1, 1)

    for (s, e, g) in spans:
        y_center_frac = ((s + e - 1) / 2) / n_positions
        ax.text(
            -0.62,
            y_center_frac,
            g,
            transform=ax.transAxes,
            ha='right',
            va='center',
            fontsize=9,
            rotation=90,
            fontweight='bold',
            color='black',
            clip_on=False
        )

    ax.tick_params(axis='x', labelsize=9)
    ax.tick_params(axis='y', labelsize=9)

    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))

    ax.set_xlabel(
        f"Average Nutrition Profile Score\n{healthy_rule_text}",
        fontsize=9
    )

    ax.set_ylabel(
        "Claims",
        fontweight='bold',
        labelpad=15,
        fontsize=9
    )

    if title:
        ax.set_title(title, fontsize=12, pad=4)

    ax.grid(
        True,
        which='both',
        axis='both',
        linestyle='--',
        linewidth=0.8,
        alpha=0.65,
        zorder=0
    )

# ============================================================
# Save single figure per beverage subcategory
# ============================================================

def save_single_subcategory_figure(subcat: str, df_sub: pd.DataFrame):

    results_df, results_melted = compute_results_for_subset(df_sub)

    subcat_folder = os.path.join(single_panel_folder, safe_name(subcat))
    os.makedirs(subcat_folder, exist_ok=True)

    healthy_text, _ = healthy_text_for_beverage_subcategory()

    n_labels = len(results_df) if not results_df.empty else 0
    fig_height = max(6, min(14, 0.35 * max(n_labels, 1) + 2))

    fig, ax = plt.subplots(figsize=(9, fig_height))

    draw_one_panel(
        ax,
        results_df,
        results_melted,
        title=f"{subcat}: Discrepancy in NP Scores by Claim",
        show_legend=True,
        healthy_rule_text=healthy_text
    )

    handles, labels = ax.get_legend_handles_labels()

    filtered = [
        (h, l)
        for h, l in zip(handles, labels)
        if l in ['With Claim', 'Without Claim']
    ]

    if filtered:
        h, l = zip(*filtered)
        ax.legend(
            h,
            l,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.08),
            ncol=2,
            fontsize=10,
            frameon=True
        )
    else:
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

    fig.tight_layout()
    fig.subplots_adjust(left=0.2)

    out_path = os.path.join(
        subcat_folder,
        f"Beverage_NewSubCategory_Discrepancy_{safe_name(subcat)}.png"
    )

    fig.savefig(out_path, dpi=600, bbox_inches='tight')
    plt.close(fig)

# ============================================================
# Save combined pages
# Page 1: first 6 subcategories, 2 columns x 3 rows
# Page 2: last subcategory, standalone panel, legend below
# ============================================================
def save_combined_panels(page_subcategories, page_idx, total_pages, df_beve_only):

    # Page 1: 2 columns × 3 rows
    # Page 2: 2 columns × 1 row, but only LEFT panel is used
    if page_idx == 0:
        rows, cols = 3, 2
        fig_w, fig_h = 12, 18
    else:
        rows, cols = 1, 2
        fig_w, fig_h = 12, 6.8

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(fig_w, fig_h),
        sharex=False,
        sharey=False
    )

    if rows * cols == 1:
        axes = [axes]
    else:
        axes = np.array(axes).flatten()

    for i in range(rows * cols):
        ax = axes[i]

        # For the last page, only draw the LEFT panel.
        # The right panel stays blank to preserve 2-column subfigure width.
        if page_idx != 0 and i == 1:
            ax.set_axis_off()
            continue

        if i >= len(page_subcategories):
            ax.set_axis_off()
            continue

        subcat = page_subcategories[i]

        df_sub = df_beve_only.loc[
            df_beve_only['NewSubCategory'] == subcat
        ].copy()

        results_df, results_melted = compute_results_for_subset(df_sub)
        healthy_text, _ = healthy_text_for_beverage_subcategory()

        draw_one_panel(
            ax,
            results_df,
            results_melted,
            title=subcat,
            show_legend=False,
            healthy_rule_text=healthy_text
        )

    # Layout
    bottom_rect = 0.10 if page_idx == 0 else 0.22

    fig.tight_layout(rect=[0, bottom_rect, 1, 0.985])

    fig.subplots_adjust(
        left=0.13,
        right=0.98,
        top=0.97,
        bottom=bottom_rect + 0.01,
        hspace=0.28 if page_idx == 0 else 0.10,
        wspace=0.80
    )

    used_axes = [
        ax for i, ax in enumerate(axes)
        if i < len(page_subcategories)
    ]

    grid_bottom = min(ax.get_position().y0 for ax in used_axes) if used_axes else 0.12

    # Dummy legend
    dummy = pd.DataFrame({
        'Average NPM Score': [0, 0],
        'Label': ['a', 'b'],
        'Claim Status': ['With Claim', 'Without Claim'],
        'Popularity': [1, 1]
    })

    dummy_ax = fig.add_subplot(111, frame_on=False)
    dummy_ax.set_axis_off()

    lg = sns.scatterplot(
        data=dummy,
        x='Average NPM Score',
        y='Label',
        hue='Claim Status',
        style='Claim Status',
        ax=dummy_ax
    )

    handles, labels = lg.get_legend_handles_labels()

    if page_idx == 0:
        legend_y = grid_bottom - 0.05
        legend_x = 0.5
    else:
        # Put legend below the LEFT standalone panel
        legend_y = grid_bottom - 0.12
        legend_x = 0.29

    fig.legend(
        handles,
        labels,
        loc='upper center',
        ncol=2,
        fontsize=11,
        frameon=True,
        borderaxespad=0,
        bbox_to_anchor=(legend_x, legend_y)
    )

    dummy_ax.remove()

    if page_idx == 0:
        out_name = (
            f"Combined_Beverage_NewSubCategory_Page_"
            f"{page_idx + 1}_2cols_3rows_of_{total_pages}.png"
        )
    else:
        out_name = (
            f"Combined_Beverage_NewSubCategory_Page_"
            f"{page_idx + 1}_left_panel_only_of_{total_pages}.png"
        )

    out_path = os.path.join(beverage_appendix_folder, out_name)

    fig.savefig(out_path, dpi=600, bbox_inches='tight')
    plt.close(fig)

# ============================================================
# MAIN
# ============================================================

df_beve_only = df.loc[df['NewCategory'] == 'Beverages'].copy()

all_subcategories = (
    df_beve_only['NewSubCategory']
    .dropna()
    .astype(str)
    .sort_values()
    .unique()
    .tolist()
)

print("Beverage NewSubCategory values:")
print(all_subcategories)

# 1) Per-subcategory single figures
for subcat in tqdm(all_subcategories, desc="Generating beverage subcategory charts"):
    df_sub = df_beve_only.loc[
        df_beve_only['NewSubCategory'] == subcat
    ].copy()

    save_single_subcategory_figure(subcat, df_sub)

# 2) Combined pages
pages = [
    all_subcategories[:6],
    all_subcategories[6:7]
]

pages = [p for p in pages if len(p) > 0]
total_pages = len(pages)

for page_idx, page_subcategories in tqdm(
    list(enumerate(pages)),
    desc="Building combined beverage subcategory pages",
    total=total_pages
):
    save_combined_panels(
        page_subcategories=page_subcategories,
        page_idx=page_idx,
        total_pages=total_pages,
        df_beve_only=df_beve_only
    )

print("\nDone.")
print(f"Single-subcategory figures saved to: {single_panel_folder}")
print(f"Combined beverage pages saved to: {beverage_appendix_folder}")

## Extended Data Table 1: Discrepancy Analysis Table for Food

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

# ============================================================
# Supplementary Table S3
# Classification of claim discrepancy across food categories
# ============================================================

os.makedirs(RESULT_PATH, exist_ok=True)

# ------------------------------------------------------------
# 0. Food sample
# ------------------------------------------------------------
df_food = df.loc[df['NewCategory'] != 'Beverages'].copy()

# ------------------------------------------------------------
# 1. Claim groups and labels
# ------------------------------------------------------------
claim_groups = {
    'Minus': [
        'LessCalorie', 'NoAddedSugar', 'SugarFree', 'LowSugar', 'Diet',
        'LessSodium', 'LessCarb', 'LessFat', 'LessTransFat', 'LessSatFat',
        'LessChol', 'LessGlycemic'
    ],
    'Plus': [
        'PlusVitamin', 'HighProtein', 'AddedCalcium', 'HighFiber'
    ],
    'Natural': [
        'AllNatural', 'NoArtAdditives', 'NoArtColourings', 'NoArtFlavourings',
        'NoArtPreservatives', 'NoAdditivesPreservatives', 'GMOFree', 'Organic', 'Wholegrain'
    ],
    'Functional': [
        'BrainNervSystem', 'ImmuneSystem', 'Digestive', 'Probiotic',
        'Antioxidant', 'WgtMuscle', 'Cardiovascular', 'BoneSkinHairEyeHealth'
    ]
}

claim_labels = {
    'LessCalorie': 'Low/No/Reduced Calorie',
    'NoAddedSugar': 'No Added Sugar',
    'SugarFree': 'Sugar Free',
    'LowSugar': 'Low/Reduced Sugar',
    'Diet': 'Diet/Light',
    'LessSodium': 'Low/No/Reduced Sodium',
    'LessCarb': 'Low/No/Reduced Carb',
    'LessFat': 'Low/No/Reduced Fat',
    'LessTransFat': 'Low/No/Reduced Trans Fat',
    'LessSatFat': 'Low/No/Reduced Saturated Fat',
    'LessChol': 'Low/No/Reduced Cholesterol',
    'LessGlycemic': 'Low/No/Reduced Glycemic',
    'PlusVitamin': 'Vitamin/Mineral Fortified',
    'HighProtein': 'High/Added Protein',
    'AddedCalcium': 'Added Calcium',
    'HighFiber': 'High/Added Fiber',
    'AllNatural': 'All Natural Product',
    'NoArtAdditives': 'No Added/Artificial Additives',
    'NoArtColourings': 'No Added/Artificial Colourings',
    'NoArtFlavourings': 'No Added/Artificial Flavourings',
    'NoArtPreservatives': 'No Added/Artificial Preservatives',
    'NoAdditivesPreservatives': 'No Additives/Preservatives',
    'GMOFree': 'GMO Free',
    'Organic': 'Organic',
    'Wholegrain': 'Whole Grain',
    'BrainNervSystem': 'Brain & Nervous System',
    'ImmuneSystem': 'Immune System',
    'Digestive': 'Digestive',
    'Probiotic': 'Probiotic/Prebiotic',
    'Antioxidant': 'Antioxidant',
    'WgtMuscle': 'Weight & Muscle Gain',
    'Cardiovascular': 'Cardiovascular',
    'BoneSkinHairEyeHealth': 'Bone/Skin/Nails/Hair/Eye Health'
}

claims = [c for grp in claim_groups.values() for c in grp]

def get_claim_type(claim):
    for g, items in claim_groups.items():
        if claim in items:
            return g
    return 'Other'

food_cats = (
    df_food['NewCategory']
    .dropna()
    .astype(str)
    .sort_values()
    .unique()
    .tolist()
)

# ------------------------------------------------------------
# 2. Helper
# ------------------------------------------------------------
def get_status(with_vals, without_vals):
    with_vals = pd.Series(with_vals).dropna()
    without_vals = pd.Series(without_vals).dropna()

    if len(with_vals) == 0 or len(without_vals) == 0:
        return "Not testable", np.nan

    diff = with_vals.mean() - without_vals.mean()

    try:
        _, p = mannwhitneyu(with_vals, without_vals, alternative='two-sided')
    except ValueError:
        return "Not testable", diff

    if pd.isna(p) or p >= 0.05:
        return "Not significant", diff

    if diff > 0:
        return "Discrepant", diff
    elif diff < 0:
        return "Aligned", diff
    else:
        return "Not significant", diff

# ------------------------------------------------------------
# 3. Build table
# 4-way classification:
# - Market-wide and within-category discrepancy
# - Market-wide discrepancy only
# - Within-category discrepancy only
# - No discrepancy
# ------------------------------------------------------------
rows = []

for claim in claims:
    if claim not in df_food.columns:
        continue

    # pooled food result
    pooled_with = df_food.loc[df_food[claim] == 1, 'np_score']
    pooled_without = df_food.loc[df_food[claim] == 0, 'np_score']
    pooled_status, _ = get_status(pooled_with, pooled_without)

    # within food category results
    discrepant_cats = []

    for cat in food_cats:
        df_cat = df_food.loc[df_food['NewCategory'] == cat].copy()

        cat_with = df_cat.loc[df_cat[claim] == 1, 'np_score']
        cat_without = df_cat.loc[df_cat[claim] == 0, 'np_score']

        cat_status, _ = get_status(cat_with, cat_without)

        if cat_status == "Discrepant":
            discrepant_cats.append(cat)

    pooled_discrepant = (pooled_status == "Discrepant")
    within_discrepant = (len(discrepant_cats) > 0)

    if pooled_discrepant and within_discrepant:
        group = "Market-wide and within-category discrepancy"
        cats_text = "; ".join(discrepant_cats)
    elif pooled_discrepant and not within_discrepant:
        group = "Market-wide discrepancy only"
        cats_text = "None"
    elif (not pooled_discrepant) and within_discrepant:
        group = "Within-category discrepancy only"
        cats_text = "; ".join(discrepant_cats)
    else:
        group = "No discrepancy"
        cats_text = ""

    rows.append({
        'Claim type': get_claim_type(claim),
        'Specific claim': claim_labels[claim],
        'Discrepancy group': group,
        'Categories with discrepancy': cats_text
    })

table_s3 = pd.DataFrame(rows)

# Keep only non-"No discrepancy" claims, then add one summary row
table_keep = table_s3.loc[table_s3['Discrepancy group'] != 'No discrepancy'].copy()

summary_row = pd.DataFrame([{
    'Claim type': '',
    'Specific claim': 'All remaining food claims',
    'Discrepancy group': 'No discrepancy',
    'Categories with discrepancy': ''
}])

table_s3_final = pd.concat([table_keep, summary_row], ignore_index=True)

group_order = {
    'Market-wide and within-category discrepancy': 1,
    'Market-wide discrepancy only': 2,
    'Within-category discrepancy only': 3,
    'No discrepancy': 4
}
type_order = {'Minus': 1, 'Plus': 2, 'Natural': 3, 'Functional': 4, '': 5}

table_s3_final['group_order'] = table_s3_final['Discrepancy group'].map(group_order)
table_s3_final['type_order'] = table_s3_final['Claim type'].map(type_order)

table_s3_final = table_s3_final.sort_values(
    by=['group_order', 'type_order', 'Specific claim'],
    ascending=[True, True, True]
).drop(columns=['group_order', 'type_order'])

csv_path = os.path.join(RESULT_PATH, "Extend_Table_1_Food_Claim_Classification.csv")
xlsx_path = os.path.join(RESULT_PATH, "Extend_Table_1_Food_Claim_Classification.xlsx")

table_s3_final.to_csv(csv_path, index=False)
table_s3_final.to_excel(xlsx_path, index=False)

print("Saved Extend Table 1:")
print(csv_path)
print(xlsx_path)
print(table_s3_final)

## Extended Data Table 2: Discrepancy Analysis Table for Beverages

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

# ============================================================
# Supplementary Table S4
# Classification of claim discrepancy across beverage subcategories
# ============================================================

os.makedirs(RESULT_PATH, exist_ok=True)

# ------------------------------------------------------------
# 0. Beverage sample
# ------------------------------------------------------------
df_beve = df.loc[df['NewCategory'] == 'Beverages'].copy()

# ------------------------------------------------------------
# 1. Claim groups and labels
# ------------------------------------------------------------
claim_groups = {
    'Minus': [
        'LessCalorie', 'NoAddedSugar', 'SugarFree', 'LowSugar', 'Diet',
        'LessSodium', 'LessCarb', 'LessFat', 'LessTransFat', 'LessSatFat',
        'LessChol', 'LessGlycemic'
    ],
    'Plus': [
        'PlusVitamin', 'HighProtein', 'AddedCalcium', 'HighFiber'
    ],
    'Natural': [
        'AllNatural', 'NoArtAdditives', 'NoArtColourings', 'NoArtFlavourings',
        'NoArtPreservatives', 'NoAdditivesPreservatives', 'GMOFree', 'Organic', 'Wholegrain'
    ],
    'Functional': [
        'BrainNervSystem', 'ImmuneSystem', 'Digestive', 'Probiotic',
        'Antioxidant', 'WgtMuscle', 'Cardiovascular', 'BoneSkinHairEyeHealth'
    ]
}

claim_labels = {
    'LessCalorie': 'Low/No/Reduced Calorie',
    'NoAddedSugar': 'No Added Sugar',
    'SugarFree': 'Sugar Free',
    'LowSugar': 'Low/Reduced Sugar',
    'Diet': 'Diet/Light',
    'LessSodium': 'Low/No/Reduced Sodium',
    'LessCarb': 'Low/No/Reduced Carb',
    'LessFat': 'Low/No/Reduced Fat',
    'LessTransFat': 'Low/No/Reduced Trans Fat',
    'LessSatFat': 'Low/No/Reduced Saturated Fat',
    'LessChol': 'Low/No/Reduced Cholesterol',
    'LessGlycemic': 'Low/No/Reduced Glycemic',
    'PlusVitamin': 'Vitamin/Mineral Fortified',
    'HighProtein': 'High/Added Protein',
    'AddedCalcium': 'Added Calcium',
    'HighFiber': 'High/Added Fiber',
    'AllNatural': 'All Natural Product',
    'NoArtAdditives': 'No Added/Artificial Additives',
    'NoArtColourings': 'No Added/Artificial Colourings',
    'NoArtFlavourings': 'No Added/Artificial Flavourings',
    'NoArtPreservatives': 'No Added/Artificial Preservatives',
    'NoAdditivesPreservatives': 'No Additives/Preservatives',
    'GMOFree': 'GMO Free',
    'Organic': 'Organic',
    'Wholegrain': 'Whole Grain',
    'BrainNervSystem': 'Brain & Nervous System',
    'ImmuneSystem': 'Immune System',
    'Digestive': 'Digestive',
    'Probiotic': 'Probiotic/Prebiotic',
    'Antioxidant': 'Antioxidant',
    'WgtMuscle': 'Weight & Muscle Gain',
    'Cardiovascular': 'Cardiovascular',
    'BoneSkinHairEyeHealth': 'Bone/Skin/Nails/Hair/Eye Health'
}

claims = [c for grp in claim_groups.values() for c in grp]

def get_claim_type(claim):
    for g, items in claim_groups.items():
        if claim in items:
            return g
    return 'Other'

subcats = (
    df_beve['NewSubCategory']
    .dropna()
    .astype(str)
    .sort_values()
    .unique()
    .tolist()
)

# ------------------------------------------------------------
# 2. Helper
# ------------------------------------------------------------
def get_status(with_vals, without_vals):
    with_vals = pd.Series(with_vals).dropna()
    without_vals = pd.Series(without_vals).dropna()

    if len(with_vals) == 0 or len(without_vals) == 0:
        return "Not testable", np.nan

    diff = with_vals.mean() - without_vals.mean()

    try:
        _, p = mannwhitneyu(with_vals, without_vals, alternative='two-sided')
    except ValueError:
        return "Not testable", diff

    if pd.isna(p) or p >= 0.05:
        return "Not significant", diff

    if diff > 0:
        return "Discrepant", diff
    elif diff < 0:
        return "Aligned", diff
    else:
        return "Not significant", diff

# ------------------------------------------------------------
# 3. Build table
# 4-way classification:
# - Market-wide and within-category discrepancy
# - Market-wide discrepancy only
# - Within-category discrepancy only
# - No discrepancy
# ------------------------------------------------------------
rows = []

for claim in claims:
    if claim not in df_beve.columns:
        continue

    pooled_with = df_beve.loc[df_beve[claim] == 1, 'np_score']
    pooled_without = df_beve.loc[df_beve[claim] == 0, 'np_score']
    pooled_status, _ = get_status(pooled_with, pooled_without)

    discrepant_subcats = []

    for subcat in subcats:
        df_sub = df_beve.loc[df_beve['NewSubCategory'] == subcat].copy()

        sub_with = df_sub.loc[df_sub[claim] == 1, 'np_score']
        sub_without = df_sub.loc[df_sub[claim] == 0, 'np_score']

        sub_status, _ = get_status(sub_with, sub_without)

        if sub_status == "Discrepant":
            discrepant_subcats.append(subcat)

    pooled_discrepant = (pooled_status == "Discrepant")
    within_discrepant = (len(discrepant_subcats) > 0)

    if pooled_discrepant and within_discrepant:
        group = "Market-wide and within-category discrepancy"
        subcats_text = "; ".join(discrepant_subcats)
    elif pooled_discrepant and not within_discrepant:
        group = "Market-wide discrepancy only"
        subcats_text = "None"
    elif (not pooled_discrepant) and within_discrepant:
        group = "Within-category discrepancy only"
        subcats_text = "; ".join(discrepant_subcats)
    else:
        group = "No discrepancy"
        subcats_text = ""

    rows.append({
        'Claim type': get_claim_type(claim),
        'Specific claim': claim_labels[claim],
        'Discrepancy group': group,
        'Categories with discrepancy': subcats_text
    })

table_s4 = pd.DataFrame(rows)

# Keep only non-"No discrepancy" claims, then add one summary row
table_keep = table_s4.loc[table_s4['Discrepancy group'] != 'No discrepancy'].copy()

summary_row = pd.DataFrame([{
    'Claim type': '',
    'Specific claim': 'All remaining beverage claims',
    'Discrepancy group': 'No discrepancy',
    'Categories with discrepancy': ''
}])

table_s4_final = pd.concat([table_keep, summary_row], ignore_index=True)

group_order = {
    'Market-wide and within-category discrepancy': 1,
    'Market-wide discrepancy only': 2,
    'Within-category discrepancy only': 3,
    'No discrepancy': 4
}
type_order = {'Minus': 1, 'Plus': 2, 'Natural': 3, 'Functional': 4, '': 5}

table_s4_final['group_order'] = table_s4_final['Discrepancy group'].map(group_order)
table_s4_final['type_order'] = table_s4_final['Claim type'].map(type_order)

table_s4_final = table_s4_final.sort_values(
    by=['group_order', 'type_order', 'Specific claim'],
    ascending=[True, True, True]
).drop(columns=['group_order', 'type_order'])

csv_path = os.path.join(RESULT_PATH, "Extend_Table_2_Beverage_Claim_Classification.csv")
xlsx_path = os.path.join(RESULT_PATH, "Extend_Table_2_Beverage_Claim_Classification.xlsx")

table_s4_final.to_csv(csv_path, index=False)
table_s4_final.to_excel(xlsx_path, index=False)

print("Saved Extend Table 2:")
print(csv_path)
print(xlsx_path)
print(table_s4_final)